# Notebook 03 - Feature Engineering (Customer-Level)

**Input:** `data/interim/transactions_customer_level.parquet` (~800K rows, 5,861 customers)

**Output:** `data/processed/customer_features.parquet` - one row per customer with ~25 features ready for segmentation, CLV, and churn modeling

## Why this notebook is the most important one

In ML, you'll often hear:*"better features beat better models."* Spending an extra day here saves a week of trying to squeeze accuracy out of model tuning

## The critical concept: snapshot date and temporal splits

**Wrong way:** Compute all features over the entire dateset, then try to predict "will customer churn". This **leaks the future into the features** - your features know what hasn't happened yet from the model's perspective.

**Right way:** Pick a `snapshot_date`. Compute features using ONLY data up to that date. Define the target using data AFTER that date. The model learns to predict the unknown future from the known past - exactly how it will operate when deployed.

Our data ends `2011-12-09`. We'll use:
- **Feature window:** all data up to `2011-09-09` (snapshot date)
- **Target window:** `2011-09-10` to `2011-12-09` (next 90 days)

Customers who first purchased AFTER the snapshot date can't be features-engineered (no history). We exclude them here.

## Feature groups we'll build

1. **RFM** — Recency, Frequency, Monetary (the foundational triad)
2. **Behavioral** — AOV, basket size, days between orders, consistency
3. **Temporal/trend** — recent vs. historical activity, momentum
4. **Product mix** — diversity, category breadth (proxied by stock-code count)
5. **Geography** — country, is_uk flag
6. **Return behavior** — return rate, return count (from cancellations table)
7. **Targets** — for downstream supervised tasks:
   - `target_revenue_90d` (regression target for CLV)
   - `target_purchased_90d` (binary, classification target for churn)